## Implement Swing-up code for initial pendulum stabilization.

In [ ]:
%reset

In [ ]:
##IMPORTING LIBRARIES
import time
import random
import numpy as np
import pandas as pd
import seaborn as sns
from numpy.linalg import inv
import matplotlib.pyplot as plt

In [ ]:
## INITIALIZES THE VARIABLES
SimulateTime = []
RealTime = []
Error = []
X1 = []
X3 = []

NewTime = .05
t = .05

k = ((2 * np.pi) / 1024)
Flag = 1

## MAXIMUM MOVEMENT DISPLACEMENT
Stroke = input('[INPUT] Enter the displacement limit value: ')

def flag(Position,Stroke,Flag):
    if Position >= Stroke and Flag == 1:
        Flag = -1
    elif Position <= -Stroke and Flag == -1:
        Flag = 1
    return Flag

c = ModbusClient(host="192.168.15.150", port=502, auto_open=True, auto_close=True)

## READING THE FIRST POSITION OF THE PLC
encoder_position = c.read_holding_registers(0, 4)
x3 = (float(struct.unpack('>h', struct.pack('>H', encoder_position[0]))[0]) * k)

print("[INFO] Test Started")
startTime = time.time()
while x3 <= 0:

    ## READING THE POSITION OF THE CLP
    encoder_position = c.read_holding_registers(0, 4)
    x3 = (float(struct.unpack('>h', struct.pack('>H', encoder_position[0]))[0]) * k)
    x1 = (float(struct.unpack('>h', struct.pack('>H', encoder_position[2]))[0]) * 0.0001)

    ## STORES THE STATES
    X1.append(x1)
    X3.append(x3)

    ## REVERSAL OF THE DIRECTION WHEN CLOSE TO THE LIMIT
    Flag = flag(x3,Stroke,Flag)

    ## SENDS THE ACTION CONTROL TO THE PLC
    if Flag == 1:
        c.write_multiple_registers(10, [struct.unpack('>H', struct.pack('>h', int(Stroke)))[0]])
    else:
        c.write_multiple_registers(10, [struct.unpack('>H', struct.pack('>h', int(-Stroke)))[0]])
    
    ## SAMPLE TIME CORRECTION
    time.sleep(NewTime)
    Time += t
    ElapsedTime = time.time() - startTime
    TimeError = 2 * (Time - ElapsedTime)
    NewTime += TimeError
    
    if ((NewTime) < 0):
        NewTime = 1e-05
        Error.append(ElapsedTime - Time)
        RealTime.append(ElapsedTime)
        SimulateTime.append(Time)
        
        elapsedTime = time.time() - startTime
elapsedTime = round(time.time() - startTime, 2)

## SAVE THE DATA
timestamp = datetime.now().strftime("CSV/Data_%d_%b_%y_%H-%M.csv")
dataframe = pd.DataFrame([Movement,Input, Error, RealTime, SimulateTime])
dataframe.to_csv(timestamp, index=False)

In [ ]:
## PLOTS THE GRAPH
plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
n=np.arange(0,len(Error),1)
plt.plot(n,Error,color='#E69F00', label='Error')
plt.title("Adjustments to Acquired Data")
plt.ylabel("Error")
plt.xlabel("Size")
plt.grid(True)
plt.legend()
sns.despine()
plt.subplot(2, 1, 2)
plt.plot(n,RealTime,color='r', label='Real')
plt.plot(n,SimulateTime,color='b',linestyle='--',label='Simulated')
plt.ylabel("Time")
plt.xlabel("Size")
plt.grid(True)
plt.legend()
sns.despine()
plt.show()